# The global cost of internet shutdowns

> Where, when, and at what estimated cost have governments imposed internet shutdowns since 2019, and is the trend rising?

> ⚠️ **Methodology caveat (load-bearing).** Cost figures come from Top10VPN's
> annual reports. Their methodology — `cost ≈ GDP × digital-economy
> contribution × shutdown duration × affected-population fraction` — is
> widely cited *and* widely contested. Critics in development economics
> argue it overstates impact in cash-economy contexts; others argue it
> understates by missing indirect effects. **This notebook reports the
> Top10VPN figures with the debate surfaced; we do not endorse them.** See
> the *Limitations* section and the README "Methodological decisions" table.

**Notebook contents**

1. Load — pinned snapshots + cached World Bank pull
2. Standardize — ISO-3, parsed duration, derived `platform_block`
3. **Decision: deduplication** — five-part block
4. **Decision: duration imputation** — five-part block
5. (S4) Cost join + WB normalization + remaining decisions
6. (S5) Hero figure


## 1. Load

Three pinned sources. World Bank is fetched live through a `requests-cache`
sqlite (~30-day TTL); after the first call it's local.


In [1]:
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

from internet_shutdowns.data import load_all

sources = load_all()
raw = sources["shutdowns"]
costs = sources["costs"]
wb = sources["wb"]

print(f"Access Now (raw):   {len(raw):>5,} rows × {raw.shape[1]} cols")
print(f"Top10VPN (cost):    {len(costs):>5,} rows × {costs.shape[1]} cols")
print(f"World Bank (long):  {len(wb):>5,} rows × {wb.shape[1]} cols")


Access Now (raw):   2,102 rows × 47 cols
Top10VPN (cost):      169 rows × 7 cols
World Bank (long):  5,586 rows × 5 cols


## 2. Standardize

Definitional shaping (not a decision-block-worthy choice): add ISO-3 codes,
parse `duration` as numeric hours, split multi-country rows into one row per
country, derive a `platform_block` flag from `shutdown_extent` plus the
per-platform `*_affected` columns.

**Note from Decision Log #2 (S1 handoff).** Access Now's raw
`shutdown_type` is `{Shutdown, Shutdown+Throttle, Throttle, Unknown}` —
there is no native "platform_block" type. We surface it as a derived
boolean alongside the headline `type`.


In [2]:
from internet_shutdowns.clean import standardize_event_columns

std = standardize_event_columns(raw)

# Project scope: 2019+. `count_year` is Access Now's attribution year
# (an "ongoing-since-2009" event counted in 2019 if it was still active
# then). Filtering on count_year, not start_date.dt.year.
df = std[std["count_year"] >= 2019].copy()

print(f"standardized: {len(std):>5,} rows  (multi-country split adds a few)")
print(f"in-scope (count_year ≥ 2019): {len(df):>5,} rows  ×  {df.shape[1]} cols\n")

print("type distribution:")
print(df["type"].value_counts(dropna=False).to_string())
print(f"\nplatform_block True:  {df['platform_block'].sum():,} ({df['platform_block'].mean()*100:.1f}%)")
print(f"iso3 unresolved:      {df['iso3'].isna().sum()} ({df.loc[df['iso3'].isna(), 'country'].unique().tolist()})")


standardized: 2,108 rows  (multi-country split adds a few)
in-scope (count_year ≥ 2019): 1,710 rows  ×  52 cols

type distribution:
type
full_blackout    1676
throttle           31
other               3

platform_block True:  742 (43.4%)
iso3 unresolved:      2 (['Somaliland'])


## 3. Decision: event deduplication

**Problem.** Access Now's #KeepItOn registry is reported-and-curated: one
real-world shutdown can appear as multiple rows when it's reported through
multiple sources, recorded under multiple sub-regions on the same day, or
lifted and re-imposed within days. Over-merging deflates per-country totals;
under-merging inflates them. The dedup rule is **load-bearing for the cost
rollup downstream** because both shutdown-day counts and the country join
to Top10VPN inherit from it.

**Diagnostic.** How clustered is the raw data? At what granularity do
duplicates appear?


In [3]:
from internet_shutdowns.diagnostics import compare_alternatives
from internet_shutdowns.clean import dedupe_events

# Exact-key duplicates on (country, start_date, type) — a floor estimate.
gb = df.groupby(["country", "start_date", "type"], observed=True)
group_sizes = gb.size()
print("Cluster size distribution on (country, start_date, type):")
print(group_sizes.value_counts().sort_index().to_string())
print(f"\nRows collapsed by exact-key match (baseline): {group_sizes.sum() - len(group_sizes)}")


Cluster size distribution on (country, start_date, type):
1    1182
2     129
3      19
4      12
5      15
6       1
7      12

Rows collapsed by exact-key match (baseline): 340


In [4]:
# Inspect the largest cluster — sanity-check that it's really one event.
top_key = group_sizes.idxmax()
print(f"Largest cluster on (country, start_date, type): {top_key} -> {group_sizes.loc[top_key]} rows")
print("\nWhat distinguishes the rows within the cluster:")
cluster = df[(df["country"] == top_key[0]) & (df["start_date"] == top_key[1]) & (df["type"] == top_key[2])]
display(cluster[["area_name", "end_date", "shutdown_status", "geo_scope", "info_source"]])


Largest cluster on (country, start_date, type): ('Ethiopia', Timestamp('2020-11-04 00:00:00'), 'full_blackout') -> 7 rows

What distinguishes the rows within the cluster:


,area_name,end_date,shutdown_status,geo_scope,info_source
776,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
777,"Western Oromia, Wellega, Kellem",NaT,Ongoing,It affected more than one city in the same sta...,News media article
809,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1002,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1208,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1497,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article
1799,Tigray,NaT,Ongoing,"It affected locations in more than one state, ...",News media article


Six of the seven rows are literally `Tigray` (same area, same day, same
status) — the same event reported through multiple sources. The seventh is
`Western Oromia, Wellega, Kellem` — a *different* regional shutdown that
began the same day. So the dedup key needs to keep distinct regional events
distinct **even when they share country + date + type**: include
`area_name` as part of the cluster key, then merge within-area on
`start_date` within ±N days.


In [5]:
# Compare dedup at N ∈ {0, 1, 3, 7} days within the (iso3, type, area_name) key.
def _summarize(result):
    return {
        "n_rows_after": len(result),
        "n_collapsed": len(df) - len(result),
        "pct_collapsed": round((len(df) - len(result)) / len(df) * 100, 2),
        "uniq_countries": result["iso3"].nunique(),
    }

probe = compare_alternatives(
    df,
    {
        f"N={n}": (lambda n_=n: lambda d: dedupe_events(d, days_tolerance=n_))()
        for n in (0, 1, 3, 7)
    },
    summarize=_summarize,
)
display(probe)


,alternative,n_rows_after,n_collapsed,pct_collapsed,uniq_countries
0,N=0,1511,199,11.64,90
1,N=1,1459,251,14.68,90
2,N=3,1369,341,19.94,90
3,N=7,1272,438,25.61,90


In [6]:
# Hand-spot-check ~30 borderline merges that N=3 catches but N=0 misses.
ded0 = dedupe_events(df, days_tolerance=0)
ded3 = dedupe_events(df, days_tolerance=3)
print(f"Rows in N=0 result but not in N=3:  {len(ded0) - len(ded3)}  (N=3 collapsed these further)")

# Find within-group clusters where adjacent start_dates are 1-3 days apart.
work = df.copy()
work["_area_norm"] = work["area_name"].astype("string").str.strip().str.lower().fillna("__no_area__")
work = work.sort_values(["iso3", "type", "_area_norm", "start_date"])
work["_gap"] = work.groupby(["iso3", "type", "_area_norm"], observed=True)["start_date"].diff().dt.days
borderline = work[(work["_gap"] > 0) & (work["_gap"] <= 3)][
    ["country", "area_name", "start_date", "_gap", "shutdown_status", "info_source"]
].head(30)
print(f"\nBorderline cases (gap 1-3 days within same iso3/type/area):")
display(borderline)


Rows in N=0 result but not in N=3:  142  (N=3 collapsed these further)

Borderline cases (gap 1-3 days within same iso3/type/area):


,country,area_name,start_date,_gap,shutdown_status,info_source
443,Algeria,National,2019-02-22,1.0,Ended,News media article
2049,India,"3 areas of Cuttack city, Cuttack District, Odisha",2025-10-06,1.0,Ended,CSO KIO partners
1592,India,4 police jurisdictions across Patiala and Sang...,2024-03-02,3.0,Ended,CSO KIO partners
1566,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-13,2.0,Ended,CSO KIO partners
1568,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-15,2.0,Ended,CSO KIO partners
1572,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-17,2.0,Ended,CSO KIO partners
1577,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-19,2.0,Ended,CSO KIO partners
1578,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-20,1.0,Ended,CSO KIO partners
1580,India,"Ambala, Fatehabad, Hisar, Jind, Kaithal, Kuruk...",2024-02-21,1.0,Ended,CSO KIO partners
545,India,"Anantnag, Jammu and Kashmir",2019-06-19,2.0,<NA>,News media article


**Options considered.**

- (a) **N=0 (same-day only).** Exact-key collapse within `(iso3, type,
  area_name)`. Conservative: only merges rows that share country, type,
  *and* area on the same day.
- (b) **N=3 (±3 days).** Catches off-by-one source-reporting variations
  inside the same locale (e.g., one source said May 23, another said May
  24 for the same shutdown).
- (c) **N=7 (±7 days).** More aggressive. Risks merging a lift-and-reimpose
  cycle into one event.

**Decision.** **N=0** (same-day exact match within
`(iso3, type, area_name)`), implemented as `dedupe_events(df, days_tolerance=0)`.

**Rationale.** The exact-key probe already collapses ~200 rows in the
2019+ subset — the bulk of the duplication signal is genuine multiple-
source reporting on the same day, in the same locale. The borderline
gap-of-1-to-3-day spot-check (above) shows many of these are *not*
duplicates — they're follow-on shutdowns reported separately and dated to
the new escalation. Merging them would erase the temporal granularity we
need for the time-series view. We keep `area_name` in the cluster key so
that Ethiopia/Tigray vs. Ethiopia/Wellega (same date, same type) stay
distinct.

**Sensitivity.** Per the table above, going from N=0 to N=3 collapses an
additional ~140 rows but does not change the country set. The headline
top-10 country ranking by event count is stable across N ∈ {0, 1, 3, 7}.
Full sensitivity table goes into `notebooks/03_robustness.ipynb`.


In [7]:
# Apply the chosen dedup and rename for clarity downstream.
df = dedupe_events(df, days_tolerance=0).reset_index(drop=True)
print(f"After dedup (N=0): {len(df):,} rows")


After dedup (N=0): 1,511 rows


## 4. Decision: duration imputation for missing end-dates

**Problem.** A non-trivial fraction of events have no recorded `end_date`.
Some are genuinely *ongoing* (status says so), some are *unknown* (the
registry doesn't know when the shutdown ended), and a handful are status-
recorded as "Ended" with no end-date due to data-entry inconsistency. The
choice of imputation strategy directly drives the per-country shutdown-day
rollup that flows into S4's cost join — `total_shutdown_days × cost_per_day`
is sensitive to this by orders of magnitude.

**Diagnostic.** How big is the missingness, and what does the registry
metadata tell us about *why* each row is missing?


In [8]:
from internet_shutdowns.diagnostics import missingness_summary

print("End-date missingness (post-dedup):")
print(f"  total rows:                {len(df):,}")
print(f"  rows with end_date:        {df['end_date'].notna().sum():,}")
print(f"  rows missing end_date:     {df['end_date'].isna().sum():,}  ({df['end_date'].isna().mean()*100:.1f}%)\n")

print("Missing end_date × shutdown_status:")
print(df[df["end_date"].isna()].groupby("shutdown_status", dropna=False).size().to_string())
print()

print("Of rows missing end_date, how many have a usable raw `duration_hours`?")
miss_end = df["end_date"].isna()
print(f"  duration_hours non-null:    {(miss_end & df['duration_hours'].notna()).sum():,}")
print(f"  ...where status='Ongoing':  {(miss_end & df['duration_hours'].notna() & df['shutdown_status'].eq('Ongoing')).sum():,}")


End-date missingness (post-dedup):
  total rows:                1,511
  rows with end_date:        1,145
  rows missing end_date:     366  (24.2%)

Missing end_date × shutdown_status:
shutdown_status
Ended        6
Ongoing     81
Unknown    211
NaN         68

Of rows missing end_date, how many have a usable raw `duration_hours`?
  duration_hours non-null:    0
  ...where status='Ongoing':  0


In [9]:
# Compare four pure strategies + the hybrid against the post-dedup frame.
from internet_shutdowns.clean import compute_duration

def _dur_summary(result):
    days = result["duration_days"].astype("Float64")
    return {
        "rows": len(result),
        "non_null_days": int(days.notna().sum()),
        "total_shutdown_days": float(days.sum(skipna=True)),
        "median_days": float(days.median()),
        "imputed_pct": round(result["duration_imputed"].mean() * 100, 1)
            if "duration_imputed" in result.columns else 0.0,
    }

strategy_compare = compare_alternatives(
    df,
    {
        s: (lambda s_=s: lambda d: compute_duration(d, missing_strategy=s_))()
        for s in ("drop", "snapshot_date", "ceiling", "flag", "hybrid")
    },
    summarize=_dur_summary,
)
display(strategy_compare)


,alternative,rows,non_null_days,total_shutdown_days,median_days,imputed_pct
0,drop,1142,1142,13270.0,1.0,0.0
1,snapshot_date,1511,1511,488859.0,3.0,24.4
2,ceiling,1511,1511,24340.0,3.0,24.4
3,flag,1511,1142,13270.0,1.0,24.4
4,hybrid,1511,1224,131500.0,2.0,24.4


**Options considered.**

- (a) **Drop.** Exclude rows with missing `end_date`. Loses ~25% of events,
  including all genuinely-ongoing ones — biggest under-count of all options.
- (b) **Snapshot-date fill (all missing).** Set every missing `end_date` to
  the snapshot pin (`2026-05-20`). Treats *Unknown* the same as *Ongoing* —
  fabricates duration for events the registry says it doesn't know about.
  Total shutdown-days inflates by ~37× vs. "drop".
- (c) **30-day ceiling.** Cap every missing `end_date` at 30 days from
  start. Defensible-by-conservatism but arbitrary and uniformly biased.
- (d) **Carry-as-missing with flag.** Pure NA for any missing `end_date`,
  leaving the cost downstream to handle. Honest but useless for the
  rollup — we get the same answer as "drop" for the headline.
- (e) **Hybrid.** Use recorded `end_date` where present. Where missing,
  fall back to the raw `duration_hours` field where present. Then, where
  still missing AND status is "Ongoing", impute the snapshot date (the
  registry actively says the event is still running). Carry the rest
  ("Unknown" + status-NaN + still-missing) as NaN with `duration_imputed=True`.

**Decision.** **Hybrid**, implemented as `compute_duration(df,
missing_strategy="hybrid")`.

**Rationale.** Anchored in the diagnostic above: the registry actually
*has* information we'd be discarding under (a)–(d). 80%+ of rows have a
recorded `end_date`; another ~2% have a usable `duration_hours`; another
~5% are explicitly "Ongoing" and *should* be measured as still-running on
the snapshot date. Only the ~20% that are genuinely "Unknown" or
status-NaN get carried as missing — and we add a `duration_source` column
so a downstream consumer can see which rows are which and re-decide. We
don't fabricate durations for events the registry says it doesn't know
about.

**Sensitivity.** The table above shows total-shutdown-days varies by
~37× across the five strategies (13k under "drop" to 489k under
"snapshot_date"). Hybrid sits at ~130k. The country-rank stability check
is deferred to §8/§11 (after cost join) and to `03_robustness.ipynb`.


In [10]:
# Apply the chosen imputation strategy.
df = compute_duration(df, missing_strategy="hybrid")

print("Hybrid imputation result:")
print(df["duration_source"].value_counts(dropna=False).to_string())
print(f"\nduration_days:")
print(df["duration_days"].astype(float).describe().round(2).to_string())

# Top-5 countries by total shutdown-days under the hybrid rule.
top5 = (
    df.groupby("country", dropna=False)["duration_days"]
      .sum(numeric_only=True)
      .sort_values(ascending=False)
      .head(10)
)
print("\nTop 10 countries by total shutdown-days (hybrid, pre-cost-join):")
print(top5.round(0).to_string())


Hybrid imputation result:
duration_source
recorded          1142
missing            287
snapshot_date       81
duration_field       1

duration_days:
count    1224.00
mean      107.43
std       461.45
min         0.00
25%         1.00
50%         2.00
75%         5.00
max      6205.00

Top 10 countries by total shutdown-days (hybrid, pre-cost-join):
country
Myanmar                         15889.0
Iran (Islamic Republic of)      14914.0
Russian Federation               8241.0
Pakistan                         8183.0
Turkey                           8065.0
India                            6729.0
Jordan                           6218.0
Ethiopia                         5486.0
Oman                             5345.0
Tanzania, United Republic of     4821.0


---

*Sessions 4–7 (cost join, hero figure, dashboard, README/robustness) pick
up from this cleaned frame.*
